In [ ]:
'''###Referenz ⇒ Canonical Form 
- pdb ID in der pdb suchen ⇒ FASTA sequence file downloaden  
- bei SAbPred: SCALOP in Submission form die Datei hochladen 
- results: für jedes region (H1, H2, L1, L2, L3) erkennt er region Sequenz aus der gesamten Sequenz (mit Antibody und antigen) und gibt einem canonical form und median structure 
- (L1-11-A → Canonical form A für eine L1-Schleife mit 11 Aminosäuren)
- muss man für alle unsere pdb Einträge machen und dann canonical cluster nochmal im code definieren (also die pdb Einträge zuordnen)
- dann der vergleich mit V-measure'''

In [ ]:
'''# SCALOP Installation und Setup Anleitung
# 1. Projekt klonen (wenn noch nicht gemacht)
git clone https://github.com/oxpig/SCALOP.git

# 2. Conda-Umgebung erstellen
conda create -n scalop-env python=3.8 -y

# 3. Conda initialisieren (falls noch nie gemacht)
conda init
# dann Terminal neu starten
exit
# dann Terminal wieder öffnen

# 4. Umgebung aktivieren
conda activate scalop-env

# 5. Abhängigkeiten installieren
conda install -c bioconda numpy biopython -y
conda install -c bioconda hmmer -y   # funktioniert nur in WSL-Umgebung, nicht in Windows direkt! Wird für assign benötigt

# 6. SCALOP lokal installieren
pip install ./SCALOP

# 7. Kernel registrieren (für Jupyter-Notebook))
pip install ipykernel
python -m ipykernel install --user --name scalop-env --display-name "Python (scalop-env)"
# dann Jupyter Notebook öffnen und den Kernel "Python (scalop-env)" auswählen. Jetzt kann man SCALOP in Jupyter Notebooks verwenden.'''

In [ ]:
from scalop.predict import assign
import pandas as pd
import numpy as np
import os

df = pd.read_csv("data/ab_ag_vseqs.tsv", sep="\t")
regions = ["H1", "H2", "L1", "L2", "L3"]

In [ ]:
fab_lists = df[["pdb", "Hchain", "Lchain", "model", "antigen_name", "antigen_species", "VH", "VL"]].values.tolist()
df_rows = []

for fab_list in fab_lists:
    pdb = fab_list[0]
    hchain = fab_list[1]
    lchain = fab_list[2]
    model = fab_list[3]
    antigen_name = fab_list[4]
    antigen_species = fab_list[5]

    vh_seq = fab_list[6]
    vl_seq = fab_list[7]

    pdb_path = f"data/download/scalop_cdr/{pdb}"

    if not os.path.exists(pdb_path):
        os.makedirs(pdb_path)

    try:
        results_vh = assign(vh_seq, scheme="chothia", definition="chothia")[0]['outputs']
        results_vl = assign(vl_seq, scheme="chothia", definition="chothia")[0]['outputs']

        seqs_and_cfs = []
        for region in regions:
            chain_type = region[0]  #z.B. "H" aus "H1" oder "L" aus "L1".

            if chain_type == "H":
                seq = results_vh[region][1]
                cf = results_vh[region][2]
            else:
                seq = results_vl[region][1]
                cf = results_vl[region][2]

            seqs_and_cfs.append(seq)
            seqs_and_cfs.append(cf)
        df_row = [pdb, hchain, lchain, model, antigen_name, antigen_species] + seqs_and_cfs
    
    except Exception as e:
        print(f"Fehler bei {pdb}: {e}")
        df_row = [pdb, hchain, lchain, model, antigen_name, antigen_species] + [np.nan]*10
    
    df_rows.append(df_row)     

ab_ag_scalop = (
    pd.DataFrame(df_rows, columns=["pdb", "Hchain", "Lchain", "model", "antigen_name", "antigen_species", "SEQ_H1", "CF_H1", "SEQ_H2", "CF_H2", "SEQ_L1", "CF_L1", "SEQ_L2", "CF_L2", "SEQ_L3", "CF_L3"])
    .dropna(subset = ["SEQ_H1", "SEQ_H2", "SEQ_L1", "SEQ_L2", "SEQ_L3"])  # Entferne Zeilen, in denen mindestens eine CDR-Sequenz fehlt
    #.drop_duplicates(subset = ["SEQ_H1", "SEQ_H2", "SEQ_L1", "SEQ_L2", "SEQ_L3"])
    .reset_index(drop = True)
)    
    
ab_ag_scalop.to_csv('data/ab_ag_scalop.tsv', sep='\t', index=False)

In [ ]:
'''fab_lists = df[["pdb", "HChain", "Lchain", "model", "antigen_name", "antigen_species"]].values.tolist()
df_rows = []

for fab_list in fab_lists:
    pdb = fab_list[0]
    hchain = fab_list[1]
    lchain = fab_list[2]
    model = fab_list[3]
    antigen_name = fab_list[4]
    antigen_species = fab_list[5]

    vh_seq = fab_list[6]
    vl_seq = fab_list[7]

    pdb_path = f"data/download/scalop_cdr/{pdb}"

    if not os.path.exists(pdb_path):
        os.makedirs(pdb_path)

    # Heavy Chain: nur H1, H2
    try:
        results_vh = assign(vh_seq, scheme="chothia", definition="chothia")[0]['outputs']
        results_vl = assign(vl_seq, scheme="chothia", definition="chothia")[0]['outputs']

        seqs_and_cfs = []
        for region in regions:
            chain_type = region[4]  # "H" oder "L" aus "CDR_H1" usw.

            if chain_type == "H":
                seq = results_vh[region][1]
                cf = results_vh[region][2]
            else:
                seq = results_vl[region][1]
                cf = results_vl[region][2]

            seqs_and_cfs.append(seq)
            seqs_and_cfs.append(cf)
        df_row = [[pdb, hchain, lchain, model, antigen_name, antigen_species] + seqs_and_cfs]
    
    except:
        df_row = [[pdb, hchain, lchain, model, antigen_name, antigen_species] + [np.nan]*10]
    
    df_rows.append(df_row)     

ab_ag_scalop = (
    pd.DataFrame(df_rows, columns=["pdb", "Hchain", "Lchain", "model", "antigen_name", "antigen_species", )
    .dropna(subset=regions, how='any')  # Entferne Zeilen, in denen mindestens eine CDR-Sequenz fehlt
    .drop_duplicates(subset = regions)
    .reset_index(drop=True)
)    
    

# In DataFrame und speichern
df_out = pd.DataFrame(results_all)
df_out.to_csv("canonical_forms_per_antibody_full.csv", index=False)
print("Fertig! Datei gespeichert als canonical_forms_per_antibody_full.csv")'''

Fertig! Datei gespeichert als canonical_forms_per_antibody_full.csv


In [ ]:
import pandas as pd

# Datei einlesen
df = pd.read_csv("../data/canonical_forms_per_antibody_full.csv")

# Schritt 1: Alle Zeilen mit mindestens einem Missing Value entfernen
df_clean = df.dropna()

# Schritt 2: Doppelte Einträge basierend auf den SEQ_* Spalten entfernen
region_seq_columns = ['SEQ_H1', 'SEQ_H2', 'SEQ_L1', 'SEQ_L2', 'SEQ_L3']
n_before = len(df_clean)

df_unique = df_clean.drop_duplicates(subset=region_seq_columns, keep='first')
n_after = len(df_unique)
duplicates_removed = n_before - n_after

# Speichern
df_unique.to_csv("ab_ag_canonical_forms.csv", index=False)

print(f"Fertig! Es wurden {duplicates_removed} Duplikate entfernt und alle N/A entfernt.")
print(f"Neue Datei hat {n_after} eindeutige PDBs.")

In [ ]:
import sys
print(sys.executable)

c:\Users\avdh3\OneDrive\Dokumente\GitHub\group04-team04\.conda\python.exe


In [ ]:
#fasta dateien downloaden

import requests      #Modul zum Herunterladen von Daten aus dem Internet
import os            #Modul für Dateipfade und Ordnerverwaltung

def download_fasta(pdb_id, outdir="fasta_files"):
    """
    Lädt die FASTA-Sequenzdatei für einen gegebenen PDB-Eintrag
    von der RCSB PDB-Website herunter und speichert sie lokal.

    Parameter:
    - pdb_id: z.B. "1abc" (Groß-/Kleinschreibung egal)
    - outdir: Zielordner, in dem die FASTA-Dateien gespeichert werden

    Rückgabe:
    - Pfad zur gespeicherten FASTA-Datei (oder None bei Fehler)
    """

    #URL zur FASTA-Datei auf der rcsb.org-Website (liefert alle Chains)
    url = f"https://www.rcsb.org/fasta/entry/{pdb_id}/display"

    #HTTP-GET-Request an die URL schicken
    response = requests.get(url)

    #Prüfen ob der Download erfolgreich war (Statuscode 200 = OK)
    if response.status_code == 200:

        #Zielordner anlegen, falls er noch nicht existiert
        os.makedirs(outdir, exist_ok=True)

        #Speicherpfad für die Datei zusammensetzen
        fasta_path = os.path.join(outdir, f"{pdb_id}.fasta")

        #Inhalt in Datei schreiben
        with open(fasta_path, "w") as f:
            f.write(response.text)

        #Pfad zur fertigen Datei zurückgeben
        return fasta_path
        

    else:
        #Fehlerausgabe, falls Download fehlgeschlagen
        print(f"Fehler beim Herunterladen von {pdb_id} (Status: {response.status_code})")
        return None